In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for device in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(device, True)

figures = Path("figures")
figures.mkdir(exist_ok=True)

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

sample_indices = [np.where(y_train == cls)[0][0] for cls in range(10)]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index in zip(axes.ravel(), sample_indices):
    ax.imshow(x_train[index], cmap="gray")
    ax.set_title(f"{y_train[index]}: {class_names[y_train[index]]}")
    ax.axis("off")
fig.suptitle("Fashion-MNIST: One Sample from Each Class")
fig.tight_layout()
fig.savefig(figures / "sample_images.png", dpi=200, bbox_inches="tight")
plt.show()

train_counts = np.bincount(y_train, minlength=10)
test_counts = np.bincount(y_test, minlength=10)
positions = np.arange(10)
width = 0.4

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(positions - width / 2, train_counts, width, label="Training")
ax.bar(positions + width / 2, test_counts, width, label="Testing")
ax.set_xticks(positions, class_names, rotation=35, ha="right")
ax.set_ylabel("Number of images")
ax.set_title("Fashion-MNIST Class Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(figures / "class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
x_train_flat = x_train.reshape(len(x_train), 784).astype("float32") / 255.0
x_test_flat = x_test.reshape(len(x_test), 784).astype("float32") / 255.0
y_train_onehot = keras.utils.to_categorical(y_train, 10)
y_test_onehot = keras.utils.to_categorical(y_test, 10)

x_search, _, y_search, _ = train_test_split(
    x_train_flat,
    y_train,
    train_size=12000,
    stratify=y_train,
    random_state=42
)

print("Before:", x_train.shape, x_test.shape, y_train.shape, y_test.shape)
print("After:", x_train_flat.shape, x_test_flat.shape, y_train_onehot.shape, y_test_onehot.shape)
print("Search:", x_search.shape, np.bincount(y_search, minlength=10))


In [ ]:
def build_baseline():
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def build_search_model(hidden_layers=1, hidden_neurons=128, activation="relu", dropout=0.0):
    model = keras.Sequential([keras.Input(shape=(784,))])
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_neurons, activation=activation))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(10, activation="softmax"))
    return model

def evaluate_model(model, x, y):
    probabilities = model.predict(x, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    labels = np.arange(10)
    metrics = {
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, labels=labels, average="weighted", zero_division=0),
        "recall": recall_score(y, predictions, labels=labels, average="weighted", zero_division=0),
        "f1_score": f1_score(y, predictions, labels=labels, average="weighted", zero_division=0)
    }
    matrix = confusion_matrix(y, predictions, labels=labels)
    report = classification_report(
        y,
        predictions,
        labels=labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    return metrics, matrix, report, predictions


In [ ]:
baseline_model = build_baseline()
baseline_model.summary()

start = time.perf_counter()
baseline_history = baseline_model.fit(
    x_train_flat,
    y_train_onehot,
    validation_split=0.1,
    epochs=20,
    batch_size=32,
    verbose=1
)
baseline_time = time.perf_counter() - start

baseline_metrics, baseline_matrix, baseline_report, baseline_predictions = evaluate_model(
    baseline_model,
    x_test_flat,
    y_test
)

print(baseline_metrics)
print(classification_report(y_test, baseline_predictions, target_names=class_names, zero_division=0))


In [ ]:
estimator = KerasClassifier(
    model=build_search_model,
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam,
    metrics=["accuracy"],
    random_state=42,
    verbose=0
)

parameter_space = {
    "model__hidden_layers": [1, 2, 3],
    "model__hidden_neurons": [32, 64, 128, 256],
    "optimizer__learning_rate": [0.1, 0.01, 0.001],
    "batch_size": [16, 32, 64, 128],
    "epochs": [10, 20, 30],
    "optimizer": [keras.optimizers.SGD, keras.optimizers.Adam, keras.optimizers.RMSprop],
    "model__activation": ["relu", "tanh", "sigmoid"],
    "model__dropout": [0.0, 0.2, 0.5]
}

cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=parameter_space,
    n_iter=12,
    scoring="accuracy",
    cv=cross_validation,
    refit=False,
    n_jobs=1,
    random_state=42,
    return_train_score=True,
    error_score=np.nan
)

start = time.perf_counter()
search.fit(x_search, y_search)
search_time = time.perf_counter() - start

search_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
best_params = search.best_params_

print("Best score:", search.best_score_)
print("Best values:", best_params)
print("Search time:", search_time)
display(search_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_fit_time"]].head(10))


In [ ]:
optimizer_class = best_params["optimizer"]
optimizer_name = "RMSProp" if optimizer_class.__name__ == "RMSprop" else optimizer_class.__name__

optimized_model = build_search_model(
    best_params["model__hidden_layers"],
    best_params["model__hidden_neurons"],
    best_params["model__activation"],
    best_params["model__dropout"]
)

optimized_model.compile(
    optimizer=optimizer_class(learning_rate=best_params["optimizer__learning_rate"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

start = time.perf_counter()
optimized_history = optimized_model.fit(
    x_train_flat,
    y_train_onehot,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    verbose=1
)
optimized_time = time.perf_counter() - start

optimized_metrics, optimized_matrix, optimized_report, optimized_predictions = evaluate_model(
    optimized_model,
    x_test_flat,
    y_test
)

print(optimized_metrics)
print(classification_report(y_test, optimized_predictions, target_names=class_names, zero_division=0))


In [ ]:
def plot_history(key, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(range(1, len(baseline_history.history[key]) + 1), baseline_history.history[key], marker="o", markersize=2, label="Baseline")
    ax.plot(range(1, len(optimized_history.history[key]) + 1), optimized_history.history[key], marker="o", markersize=2, label="Optimized")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(figures / filename, dpi=200, bbox_inches="tight")
    plt.show()

plot_history("accuracy", "Training Accuracy vs Epoch", "Training Accuracy", "training_accuracy.png")
plot_history("val_accuracy", "Validation Accuracy vs Epoch", "Validation Accuracy", "validation_accuracy.png")
plot_history("loss", "Training Loss vs Epoch", "Categorical Cross-Entropy Loss", "training_loss.png")
plot_history("val_loss", "Validation Loss vs Epoch", "Categorical Cross-Entropy Loss", "validation_loss.png")

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(optimized_matrix, display_labels=class_names).plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)
ax.set_title("Optimized MLP Confusion Matrix")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(figures / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

top = search_results.head(10).copy()
labels = []
for _, row in top.iterrows():
    optimizer = row["param_optimizer"]
    optimizer_text = "RMSProp" if optimizer.__name__ == "RMSprop" else optimizer.__name__
    labels.append(
        f"L={int(row['param_model__hidden_layers'])}, N={int(row['param_model__hidden_neurons'])}, "
        f"{optimizer_text}, lr={row['param_optimizer__learning_rate']}, "
        f"{row['param_model__activation']}, d={row['param_model__dropout']}"
    )

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(np.arange(len(top)), top["mean_test_score"], xerr=top["std_test_score"])
ax.set_yticks(np.arange(len(top)), labels)
ax.invert_yaxis()
ax.set_xlabel("Mean 5-fold Cross-Validation Accuracy")
ax.set_title("Top RandomizedSearchCV Configurations")
fig.tight_layout()
fig.savefig(figures / "hyperparameter_search_results.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Baseline", "Optimized"], [baseline_metrics["accuracy"], optimized_metrics["accuracy"]])
ax.set_ylabel("Testing Accuracy")
ax.set_title("Baseline vs Optimized MLP Testing Accuracy")
for bar, value in zip(bars, [baseline_metrics["accuracy"], optimized_metrics["accuracy"]]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.002, f"{value:.4f}", ha="center")
fig.tight_layout()
fig.savefig(figures / "accuracy_comparison.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
best_hyperparameters = pd.DataFrame({
    "Hyperparameter": [
        "Hidden layers",
        "Hidden neurons",
        "Learning rate",
        "Batch size",
        "Optimizer",
        "Activation function",
        "Epochs",
        "Dropout",
        "Cross-validation accuracy",
        "Testing accuracy"
    ],
    "Value": [
        best_params["model__hidden_layers"],
        best_params["model__hidden_neurons"],
        best_params["optimizer__learning_rate"],
        best_params["batch_size"],
        optimizer_name,
        best_params["model__activation"],
        best_params["epochs"],
        best_params["model__dropout"],
        search.best_score_,
        optimized_metrics["accuracy"]
    ]
})
display(best_hyperparameters)

performance_comparison = pd.DataFrame([
    {
        "Model": "Baseline",
        "Accuracy": baseline_metrics["accuracy"],
        "Precision": baseline_metrics["precision"],
        "Recall": baseline_metrics["recall"],
        "F1-score": baseline_metrics["f1_score"],
        "Training time": baseline_time
    },
    {
        "Model": "Optimized",
        "Accuracy": optimized_metrics["accuracy"],
        "Precision": optimized_metrics["precision"],
        "Recall": optimized_metrics["recall"],
        "F1-score": optimized_metrics["f1_score"],
        "Training time": optimized_time
    }
])
display(performance_comparison)

classification_table = pd.DataFrame(optimized_report).T
display(classification_table)


In [ ]:
xor_x = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
xor_y = np.array([[0.0], [1.0], [1.0], [0.0]])

rng = np.random.default_rng(42)
xor_w1 = rng.normal(0, 0.8, (2, 4))
xor_b1 = np.zeros((1, 4))
xor_w2 = rng.normal(0, 0.8, (4, 1))
xor_b2 = np.zeros((1, 1))
xor_learning_rate = 4.0
xor_states = []

for epoch in range(101):
    xor_hidden = np.tanh(xor_x @ xor_w1 + xor_b1)
    xor_output = 1.0 / (1.0 + np.exp(-(xor_hidden @ xor_w2 + xor_b2)))
    xor_loss = -np.mean(xor_y * np.log(xor_output + 1e-12) + (1.0 - xor_y) * np.log(1.0 - xor_output + 1e-12))
    xor_prediction = (xor_output >= 0.5).astype(int)
    xor_accuracy = np.mean(xor_prediction == xor_y)
    xor_states.append((epoch, xor_w1.copy(), xor_b1.copy(), xor_w2.copy(), xor_b2.copy(), float(xor_loss), float(xor_accuracy)))
    if epoch > 0 and xor_accuracy == 1.0:
        break
    xor_dz2 = (xor_output - xor_y) / len(xor_x)
    xor_dw2 = xor_hidden.T @ xor_dz2
    xor_db2 = xor_dz2.sum(axis=0, keepdims=True)
    xor_dz1 = (xor_dz2 @ xor_w2.T) * (1.0 - xor_hidden * xor_hidden)
    xor_dw1 = xor_x.T @ xor_dz1
    xor_db1 = xor_dz1.sum(axis=0, keepdims=True)
    xor_w1 -= xor_learning_rate * xor_dw1
    xor_b1 -= xor_learning_rate * xor_db1
    xor_w2 -= xor_learning_rate * xor_dw2
    xor_b2 -= xor_learning_rate * xor_db2

xor_grid_x, xor_grid_y = np.meshgrid(
    np.linspace(-0.3, 1.3, 220),
    np.linspace(-0.3, 1.3, 220)
)
xor_grid = np.c_[xor_grid_x.ravel(), xor_grid_y.ravel()]

rows = int(np.ceil(len(xor_states) / 4))
fig, axes = plt.subplots(rows, 4, figsize=(12, 2.8 * rows))
axes = np.atleast_1d(axes).ravel()

for ax in axes:
    ax.axis("off")

for ax, state in zip(axes, xor_states):
    epoch, w1, b1, w2, b2, loss, accuracy = state
    hidden = np.tanh(xor_grid @ w1 + b1)
    output = 1.0 / (1.0 + np.exp(-(hidden @ w2 + b2)))
    region = (output >= 0.5).astype(int).reshape(xor_grid_x.shape)
    ax.axis("on")
    ax.contourf(xor_grid_x, xor_grid_y, region, alpha=0.2, levels=[-0.5, 0.5, 1.5])
    for cls in [0, 1]:
        points = xor_x[xor_y.ravel() == cls]
        ax.scatter(points[:, 0], points[:, 1], s=45, label=f"Class {cls}")
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("x1", fontsize=8)
    ax.set_ylabel("x2", fontsize=8)
    title = "Initialization" if epoch == 0 else f"Epoch {epoch}"
    ax.set_title(f"{title} | acc={accuracy:.2f} | loss={loss:.3f}", fontsize=8)
    ax.grid(alpha=0.18)

fig.suptitle("XOR with a Multi-Layer Perceptron: Decision Boundary by Epoch", fontsize=15)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(figures / "xor_mlp_epochs.png", dpi=200, bbox_inches="tight")
plt.show()

print("Converged epoch:", xor_states[-1][0])
print("Final XOR accuracy:", xor_states[-1][6])
